# Digital Image Forensics
## Enhancing CCTV Footage for Criminal Investigation using Image Processing Techniques

---

| | |
|---|---|
| **Team Members** | SH Nihil Mukkesh |
| | Nikhil Jones A |
| | Rohan Gubba |
| | Siddartha P.N.V |
| **Course** | Digital Image Processing |

---

### Abstract

This notebook demonstrates a forensic image enhancement workflow applied to a CCTV surveillance image from a parking lot. Using only traditional Digital Image Processing techniques — histogram equalization, spatial filtering, sharpening, bit-plane slicing, and edge detection — we systematically improve the visibility and quality of the footage to aid criminal investigation. No Machine Learning or Deep Learning models are used.

---
## 1 · Import Libraries

We use three core libraries:

| Library | Purpose |
|---|---|
| **OpenCV** (`cv2`) | Image reading, colour-space conversion, filtering, edge detection |
| **NumPy** | Fast array/matrix operations on pixel data |
| **Matplotlib** | Visualisation of images and histograms |
| **os** | Filesystem operations (creating output directories) |

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "figure.facecolor": "white",
})

print("Libraries imported successfully.")
print(f"OpenCV version : {cv2.__version__}")
print(f"NumPy  version : {np.__version__}")

---
## 2 · Read & Display the Original Image

The input is a single CCTV frame (`4.png`) captured from a parking-lot surveillance camera. Before applying any enhancement we inspect the raw image and its basic properties — resolution, height, width, and number of colour channels.

OpenCV reads images in **BGR** order, so we convert to **RGB** for correct Matplotlib display.

In [ ]:
def load_image(path: str) -> np.ndarray:
    """Load an image from disk and return it in RGB colour space."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Image not found: {path}")
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb


IMAGE_PATH = "4.png"
original = load_image(IMAGE_PATH)

height, width, channels = original.shape
print(f"Resolution : {width} × {height}")
print(f"Height     : {height} px")
print(f"Width      : {width} px")
print(f"Channels   : {channels}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(original)
ax.set_title("Original CCTV Image")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 3 · Convert to Grayscale

### Why grayscale?

Colour images contain three channels (R, G, B), tripling the data volume. Many classical DIP operations — histogram equalization, gradient-based edge detection, bit-plane slicing — are defined on **single-channel** intensity images.

**Mathematical intuition:**  
The luminance (grayscale) value is a weighted sum of the colour channels that matches human perception:

$$Y = 0.299\,R + 0.587\,G + 0.114\,B$$

### Forensic relevance

Converting to grayscale removes colour bias caused by artificial lighting (sodium-vapour lamps, LEDs) commonly found in parking lots, letting investigators focus on structural details — shapes, textures, and edges.

In [ ]:
def convert_to_gray(image: np.ndarray) -> np.ndarray:
    """Convert an RGB image to grayscale."""
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)


gray = convert_to_gray(original)
cv2.imwrite(os.path.join(OUTPUT_DIR, "grayscale.png"), gray)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale")
axes[1].axis("off")
plt.tight_layout()
plt.show()

---
## 4 · Histogram Analysis

### What is a histogram?

An image histogram counts how many pixels fall into each of the 256 intensity bins (0 – 255). It reveals:

| Pattern | Meaning |
|---|---|
| Pixels clustered near 0 | Image is **dark / underexposed** |
| Pixels clustered near 255 | Image is **bright / overexposed** |
| Narrow spread | **Low contrast** |
| Wide, uniform spread | **Good contrast** |

### Forensic relevance

CCTV cameras often capture images in poor lighting. A histogram tells us *how much* contrast improvement is needed before we apply equalization.

In [ ]:
def plot_histogram(image: np.ndarray, title: str = "Histogram",
                   color: str = "steelblue") -> None:
    """Plot the intensity histogram of a grayscale image."""
    hist = cv2.calcHist([image], [0], None, [256], [0, 256]).flatten()
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.fill_between(range(256), hist, color=color, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Frequency")
    ax.set_xlim([0, 255])
    plt.tight_layout()
    plt.show()


plot_histogram(gray, title="Histogram of Grayscale Image")

print(f"Mean intensity   : {gray.mean():.2f}")
print(f"Std deviation    : {gray.std():.2f}")
print(f"Min / Max        : {gray.min()} / {gray.max()}")

---
## 5 · Histogram Equalization

### Why?

Histogram Equalization redistributes pixel intensities so that the histogram becomes approximately **uniform**. This stretches the dynamic range and significantly improves contrast — critical for footage captured in dimly lit parking lots.

### Mathematical intuition

For an image with $L = 256$ levels and $N$ total pixels, the transformation is:

$$s_k = (L - 1) \sum_{j=0}^{k} \frac{n_j}{N}$$

where $n_j$ is the number of pixels at intensity $j$. This is essentially the **cumulative distribution function (CDF)** scaled to $[0, 255]$.

### Forensic relevance

After equalization, previously invisible details in dark regions (shadows under vehicles, faces in low light) become visible to investigators.

In [ ]:
def equalize_image(image: np.ndarray) -> np.ndarray:
    """Apply histogram equalization to a grayscale image."""
    return cv2.equalizeHist(image)


equalized = equalize_image(gray)
cv2.imwrite(os.path.join(OUTPUT_DIR, "histogram_equalized.png"), equalized)

# Before vs After
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Before Equalization")
axes[0].axis("off")
axes[1].imshow(equalized, cmap="gray")
axes[1].set_title("After Equalization")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# Histogram comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
hist_before = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
hist_after  = cv2.calcHist([equalized], [0], None, [256], [0, 256]).flatten()
axes[0].fill_between(range(256), hist_before, color="steelblue", alpha=0.7)
axes[0].set_title("Histogram — Before")
axes[0].set_xlim([0, 255])
axes[1].fill_between(range(256), hist_after, color="coral", alpha=0.7)
axes[1].set_title("Histogram — After")
axes[1].set_xlim([0, 255])
plt.tight_layout()
plt.show()

---
## 6 · Image Smoothing (Spatial Filtering)

Noise is inevitable in CCTV footage due to low-cost sensors, compression artefacts, and poor lighting. Smoothing filters suppress noise before further analysis.

### Filters used

| Filter | Kernel idea | Best for |
|---|---|---|
| **Average (Box)** | Every neighbour contributes equally: $h(x,y) = \dfrac{1}{k^2}$ | General noise reduction |
| **Gaussian** | Neighbours weighted by a 2-D Gaussian: $G(x,y) = \dfrac{1}{2\pi\sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}}$ | **Gaussian (sensor) noise** |
| **Median** | Replaces pixel with the **median** of the neighbourhood (non-linear) | **Salt-and-pepper noise** |

### Forensic relevance

Denoising the image makes subsequent steps (sharpening, edge detection) more reliable, reducing false edges caused by noise.

In [ ]:
def average_filter(image: np.ndarray, ksize: int = 5) -> np.ndarray:
    """Apply an average (box) filter."""
    return cv2.blur(image, (ksize, ksize))


def gaussian_filter(image: np.ndarray, ksize: int = 5,
                    sigma: float = 1.0) -> np.ndarray:
    """Apply a Gaussian low-pass filter."""
    return cv2.GaussianBlur(image, (ksize, ksize), sigma)


def median_filter(image: np.ndarray, ksize: int = 5) -> np.ndarray:
    """Apply a median filter."""
    return cv2.medianBlur(image, ksize)


avg   = average_filter(gray)
gauss = gaussian_filter(gray)
med   = median_filter(gray)

cv2.imwrite(os.path.join(OUTPUT_DIR, "average.png"), avg)
cv2.imwrite(os.path.join(OUTPUT_DIR, "gaussian.png"), gauss)
cv2.imwrite(os.path.join(OUTPUT_DIR, "median.png"), med)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Original (Gray)", "Average Filter", "Gaussian Filter", "Median Filter"]
images = [gray, avg, gauss, med]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("• Gaussian filter is most effective at removing Gaussian (sensor) noise.")
print("• Median filter is most effective at removing salt-and-pepper noise.")

---
## 7 · Image Sharpening

Sharpening enhances edges and fine details by amplifying high-frequency components. Two classical approaches are used:

### Laplacian Filter

The **Laplacian** is a second-order derivative operator:

$$\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$

We subtract the Laplacian from the original to sharpen:  
$g(x,y) = f(x,y) - c\,\nabla^2 f(x,y)$

### Unsharp Masking

1. Create a blurred copy $\bar{f}$ (Gaussian blur).  
2. Compute the "unsharp mask": $m = f - \bar{f}$  
3. Add it back: $g = f + k \cdot m$, where $k$ controls sharpening strength.

### Forensic relevance

Sharpening can recover blurred text on licence plates, bring out facial features, and make weapon outlines distinguishable.

In [ ]:
def laplacian_sharpen(image: np.ndarray) -> np.ndarray:
    """Sharpen an image using the Laplacian operator."""
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    sharpened = np.clip(image.astype(np.float64) - laplacian, 0, 255)
    return sharpened.astype(np.uint8)


def unsharp_mask(image: np.ndarray, ksize: int = 5,
                 sigma: float = 1.0, strength: float = 1.5) -> np.ndarray:
    """Sharpen an image using Unsharp Masking."""
    blurred = cv2.GaussianBlur(image, (ksize, ksize), sigma)
    mask = image.astype(np.float64) - blurred.astype(np.float64)
    sharpened = np.clip(image.astype(np.float64) + strength * mask, 0, 255)
    return sharpened.astype(np.uint8)


lap     = laplacian_sharpen(gray)
unsharp = unsharp_mask(gray)

cv2.imwrite(os.path.join(OUTPUT_DIR, "laplacian.png"), lap)
cv2.imwrite(os.path.join(OUTPUT_DIR, "unsharp.png"), unsharp)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ["Original (Gray)", "Laplacian Sharpened", "Unsharp Masking"]
images = [gray, lap, unsharp]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 8 · Bit-Plane Slicing

### Concept

Every grayscale pixel is an 8-bit integer (0–255). We can decompose each pixel into **8 binary planes**:

$$I(x,y) = \sum_{k=0}^{7} b_k(x,y) \cdot 2^k$$

- **Bit 0 (LSB):** Carries the least perceptual weight — mostly noise.
- **Bit 7 (MSB):** Carries the most perceptual weight — the dominant structure of the image.

### Forensic relevance

- **MSB planes (5–7)** reveal the structural content and can highlight hidden or obscured features.  
- **LSB planes (0–2)** can expose **steganographic data** — hidden messages or watermarks embedded in the image — which is a direct application in digital forensics.

In [ ]:
def bit_plane_slice(image: np.ndarray) -> list:
    """Extract all 8 bit planes from a grayscale image."""
    planes = []
    for k in range(8):
        plane = ((image >> k) & 1) * 255
        planes.append(plane.astype(np.uint8))
    return planes


planes = bit_plane_slice(gray)
cv2.imwrite(os.path.join(OUTPUT_DIR, "bitplane7.png"), planes[7])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    ax.imshow(planes[idx], cmap="gray")
    ax.set_title(f"Bit Plane {idx}")
    ax.axis("off")

plt.suptitle("Bit-Plane Slicing (Bit 0 = LSB  →  Bit 7 = MSB)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 9 · Edge Detection

### Why edges matter

Edges correspond to object boundaries — the outlines of people, vehicles, licence plates, and weapons. Detecting them is a fundamental step in object identification.

### Operators used

| Operator | Description |
|---|---|
| **Sobel X** | Approximates $\partial f / \partial x$ — detects **vertical** edges |
| **Sobel Y** | Approximates $\partial f / \partial y$ — detects **horizontal** edges |
| **Sobel Magnitude** | $M = \sqrt{G_x^2 + G_y^2}$ — combined gradient magnitude |
| **Canny** | Multi-stage detector: Gaussian smoothing → gradient → non-max suppression → hysteresis thresholding. Produces thin, well-connected edges. |

### Forensic relevance

Edge maps can isolate object contours even in low-contrast footage, helping investigators identify suspects, weapons, or vehicle silhouettes that are not obvious in the raw image.

In [ ]:
def detect_edges(image: np.ndarray,
                 canny_low: int = 50,
                 canny_high: int = 150) -> dict:
    """Apply Sobel and Canny edge detectors."""
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    sobel_x_abs = cv2.convertScaleAbs(sobel_x)
    sobel_y_abs = cv2.convertScaleAbs(sobel_y)

    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_mag = np.clip(sobel_mag, 0, 255).astype(np.uint8)

    canny = cv2.Canny(image, canny_low, canny_high)

    return {
        "sobel_x":   sobel_x_abs,
        "sobel_y":   sobel_y_abs,
        "sobel_mag": sobel_mag,
        "canny":     canny,
    }


edges = detect_edges(gray)
cv2.imwrite(os.path.join(OUTPUT_DIR, "canny.png"), edges["canny"])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
edge_titles = ["Sobel X", "Sobel Y", "Sobel Magnitude", "Canny"]
edge_imgs   = [edges["sobel_x"], edges["sobel_y"],
               edges["sobel_mag"], edges["canny"]]

for ax, img, title in zip(axes, edge_imgs, edge_titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 10 · Comparison Panel

A consolidated view of every enhancement technique applied in this notebook, enabling a quick visual comparison for investigators.

In [ ]:
def comparison_plot(images: list, titles: list,
                    cols: int = 4, figsize: tuple = (18, 9)) -> None:
    """Display multiple images in a grid with titles."""
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flat

    for idx, (img, title) in enumerate(zip(images, titles)):
        if img.ndim == 3:
            axes[idx].imshow(img)
        else:
            axes[idx].imshow(img, cmap="gray")
        axes[idx].set_title(title)
        axes[idx].axis("off")

    for idx in range(len(images), len(list(axes))):
        fig.delaxes(axes[idx])

    plt.suptitle("Enhancement Techniques — Comparison Panel",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


comparison_images = [
    original, equalized, gauss, med,
    lap, unsharp, planes[7], edges["canny"],
]

comparison_titles = [
    "Original", "Histogram Equalized", "Gaussian Filter", "Median Filter",
    "Laplacian Sharpened", "Unsharp Masking", "Bit Plane 7 (MSB)", "Canny Edge Detection",
]

comparison_plot(comparison_images, comparison_titles)

---
## 11 · Forensic Discussion

Surveillance cameras are ubiquitous in modern urban environments — parking lots, highways, ATM vestibules, and building perimeters. However, the footage they produce is frequently degraded by poor lighting, low-resolution sensors, lossy compression, and motion blur. When such footage becomes evidence in a criminal investigation, raw frames are often inadequate for identifying suspects, reading licence plates, or spotting critical objects such as weapons.

**Digital image enhancement bridges this gap.** By applying well-understood, mathematically grounded techniques, forensic analysts can systematically improve visibility without introducing fabricated content — a crucial requirement for evidence admissibility.

### How each technique aids investigation

| Technique | Investigative Benefit |
|---|---|
| **Histogram Equalization** | Reveals details hidden in shadows — faces under car canopies, objects on a dark floor — by stretching the contrast range. |
| **Gaussian / Median Filtering** | Suppresses sensor noise and compression artefacts, producing a cleaner base image for analysis. Median filtering is particularly effective against salt-and-pepper noise caused by transmission errors. |
| **Laplacian Sharpening & Unsharp Masking** | Enhances edges and fine textures — the strokes on a number plate, a tattoo, or the outline of a concealed object. |
| **Bit-Plane Slicing** | The MSB planes isolate major structural content, which can be useful when the image is heavily noisy. The LSB planes may reveal steganographic information — hidden data embedded by a perpetrator or a surveillance watermark for authentication. |
| **Edge Detection (Sobel / Canny)** | Produces boundary maps that highlight object silhouettes — useful for counting persons, detecting weapons, or tracing vehicle contours even when the interior detail is washed out. |

### Practical forensic examples

1. **Suspect Identification:** Histogram equalization followed by sharpening can make a blurry face recognisable enough for an eyewitness comparison.
2. **Weapon Detection:** Edge detection can outline a firearm or knife partially concealed by clothing or shadow.
3. **Visibility Improvement:** Night-time footage with extremely low contrast becomes usable after equalization and noise reduction.
4. **Vehicle Number Plate Reading:** Unsharp masking can recover text rendered illegible by motion blur or defocus.
5. **Evidence Integrity:** Bit-plane analysis can verify whether an image has been tampered with by checking for inconsistencies in the LSB planes.

It is important to emphasise that these techniques **do not fabricate information**. They operate on existing pixel data, redistributing intensities, suppressing noise, and highlighting gradients. This makes them forensically defensible — a key consideration when evidence is presented in court. Every operation is deterministic and reproducible, ensuring that two analysts applying the same pipeline to the same image will obtain identical results.

In summary, traditional DIP provides a reliable, transparent, and computationally efficient toolkit for law enforcement agencies to extract actionable intelligence from surveillance footage.

---
## 12 · Conclusion

### Summary

This notebook demonstrated a complete forensic image enhancement pipeline using traditional Digital Image Processing techniques on a CCTV frame from a parking lot. Starting from a raw surveillance image, we applied:

- **Grayscale conversion** to simplify processing.
- **Histogram analysis and equalization** to improve contrast.
- **Spatial smoothing filters** (Average, Gaussian, Median) to suppress noise.
- **Sharpening operators** (Laplacian, Unsharp Masking) to enhance edges and details.
- **Bit-plane slicing** to isolate structural information and detect potential steganography.
- **Edge detection** (Sobel, Canny) to extract object boundaries.

### Advantages

| # | Advantage |
|---|---|
| 1 | **No training data required** — works on any image without model fitting. |
| 2 | **Deterministic and reproducible** — same input always yields same output. |
| 3 | **Computationally lightweight** — runs in real time on standard hardware. |
| 4 | **Forensically defensible** — no content fabrication; operations are transparent. |
| 5 | **Widely understood** — the mathematical foundations are well established and accepted in court. |

### Limitations

| # | Limitation |
|---|---|
| 1 | Cannot recover information that was never captured (e.g., completely saturated regions). |
| 2 | Enhancement may amplify artefacts alongside genuine details. |
| 3 | Parameter selection (kernel size, thresholds) is subjective and may vary across images. |
| 4 | Does not perform semantic understanding — cannot identify *what* an object is, only improve its visibility. |

### Future Improvements

- **Adaptive histogram equalization (CLAHE)** for locally varying contrast.
- **Frequency-domain filtering** (Fourier, Wavelet) for more selective noise removal.
- **Super-resolution techniques** to increase effective image resolution.
- **Video-level processing** — leveraging temporal redundancy across multiple frames.
- **Integration with object detection systems** for automated suspect and vehicle identification (outside the scope of classical DIP).

In [ ]:
print("=" * 60)
print("  All processed images saved to the 'outputs/' directory.")
print("=" * 60)

for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  ✓ {fname:<30s} ({size_kb:.1f} KB)")

print("=" * 60)
print("  Notebook execution complete.")
print("=" * 60)